# C3 Aerodynamics of airfoil in a 2D cascade

## Introduction

In the last session, the simulation of the NACA 65(1)-212 airfoil for 4 degrees of incidence angle was studied, with special focus on boundary layer resolution and grid convergence study. We adopt the medium mesh, and we study the behavior of the airfoil for several incidence angles, ranging from 0 to 12 degrees.

As a major upgrade, we introduce also the simulation of the **2D cascade**, as studied in the [`D2_mean_line_analysis_and_2d_cascade`](../1_Design/D2_mean_line_analysis_and_2d_cascade.ipynb) notebook. The 2D cascade simulated here is similar to the one from the S&P axial fan, but the airfoil profile 
is still our NACA, and it is not cambered.

## Learning objectives

1. **Simulate** the aerodynamics of an airfoil in a 2D cascade for several attack angles.
2. **Discuss** the behavior of the airfoil for different flow rates
3. **Compute** Diffusion Factor


## Previous tasks
[] Study the [`D2_mean_line_analysis_and_2d_cascade`](../1_Design/D2_mean_line_analysis_and_2d_cascade.ipynb) notebook and refresh the basic concepts of mean line analysis

## Steps for the simulation of the 2D cascade

1. We make a copy of previous project and name it `NACA 65(1)-212 2D cascade`.
2. In this new project, delete all the runs and meshes. We are going to make a fresh start, using only the original geometry,
3. In `Simscale` it is possible to make some simple operations (as scale and rotate) on the CAD geometry. Scale the geometry, with a factor of 0.2 (chord will be $l = 60\,\text{mm}$), and rotate it, along the $z$ axis, the stagger angle $\xi = 50^\circ$. 
4. In the 2D cascade the $x$ axis is the axial direction of the fan, not the relative direction of flow. Also, in our case, the airfoil chord (300 mm) is huge for an axial fan. The first step, hence, is to scale and rotate
   ![CAD_transform](./images/C3_rotate_scale.png)
5. We are going to consider that the pitch is $s = 120 \,\text{mm}$ so that the solidity is $\sigma = 0.5$. The domain will be in the $y$ direction from $-s/2$ to $s/2$, end the boundary conditions in these places will be _periodic_, to simulate the presence of other blades in the $y$ direction.
6. Make an `Hex-dominant parametric` mesh with a discretization of 80x8x1 with a background box extending from (-0.3,-0.06,0) to (0.9,0.06,0.06). Check also that the `Material Point` is inside the background box, but outside the geometry.
7. To get a more accurate result around the airfoil and the wake, define a `Refinement box` primitive with bounding box from (-0.05,-0.06,0) to (0.15,0.06,0.06). 
8. The discretization of the main background mesh will be 100x10x1. The `Refinement box` level will be 2. Define also a boundary layer mesh around the airfoil with 2 layers, expansion ratio 1.2, and final layer thickness 0.5. The resulting mesh should be like this:
    ![Mesh](./images/C3_mesh.png)
    ![Mesh_detail](./images/C3_mesh_detail.png)
9. Define the material "Air" for the whole domain.
10. Define the boundary conditions:
    1.  The patches normal to $z$ direction are `symmetry`, like in the previous session with isolated airfoil
    2.  The airfoil is `wall`, with `wall function` for computation of boundary layer
    3.  The patch of minimum $x$ is `Velocity inlet` with velocity (10,13.76,0). This velocity will have an incidence angle of $4^\circ$, since the angle with respect the $x$ axis is $54^\circ$, 4 more than the stagger angle
    4.  The patch of maximum $x$ is `Pressure oulet` with an static pressure of $0 \,\text{Pa}$
    5.  The patches for minimum and maximum $y$ will be `Periodic`. That means that the solution obtained in both patches will be same, simulating a cycling domain.
11. Define monitors for forces and moments in the airfoil and the computation of $y^+$. Define a monitor to measure the average pressure in the inlet patch.
12. Run the simulation. Name it `alpha_4`, with reference to the incidence angle.

## Post-processing

1. We know the value of $\beta_1 = 54^\circ$ since it is an input. But the value of $\beta_2$ has to be measured. We are going to compute the value of $\beta_2$ from the distribution of $x$ and $y$ velocities downstream. Not too close to the trailing edge and neither too far away. Typically, about 20% to 50% of chord length is used. We are going to make this measurement about 20 mm downstream, 30% of chord length (showing the mesh can be helpful to pick the points), and save both velocities in `CSV` files. You can fins the files `C3_alpha_4_W2x.csv` and `C3_alpha_4_W2y.csv` in the `data` folder.
   ![plot over line](./images/C3_plot_over_line.png)
   ![velocity CSV](./images/C3_save_velocity_CSV.png)
   The angle $\beta_2$ has to be computed as mass weighted average of $x$ and $y$ velocities,
   $$ \tan(\beta_2) = \frac{ \overline{W_{2y}}} {\overline{W_{2x} } }  $$
   where 
   $$ \overline{W_{2x}} = \frac{\int{W_{2x}^2}\text{d}y }{ \int{W_{2x}}\text{d}y }$$
   $$ \overline{W_{2y}} = \frac{\int{W_{2y} W_{2x} }\text{d}y }{ \int{W_{2x}}\text{d}y }$$
   

In [1]:
import pandas as pd
from resources.utils import compute_bulk_beta2_from_distribution, compute_cascade_coefficients

In [2]:
df_x = pd.read_csv('data/C3_alpha_4_W2x.csv')
df_y = pd.read_csv('data/C3_alpha_4_W2y.csv')

In [3]:
y_points = df_x['Distance along the path (m)'].values
W2x_points = df_x['Velocity X (m/s)'].values
W2y_points = df_y['Velocity Y (m/s)'].values

In [4]:
compute_bulk_beta2_from_distribution(y_points, W2x_points, W2y_points)

Computed bulk mass-weighted outlet flow angle: beta2 = 50.21 degrees
Computed bulk mass-weighted axial velocity: W2x_mass = 10.01 m/s
Computed bulk mass-weighted tangential velocity: W2y_mass = 12.01 m/s


(np.float64(50.20877782174788),
 np.float64(10.007092156803784),
 np.float64(12.014627393654036))

In [5]:
F_x = -0.175 # N
F_y = 0.153 # N
c_x = 10 # m/s
U_y1 = 13.76  # m/s
rho =1.2 # kg/m^3
s = 0.12 # m
l = 0.06 # m
b = 0.06 # m
beta_2 = 50.21 # degrees

compute_cascade_coefficients(F_x=F_x, F_y=F_y, c_x=c_x, U_y1=U_y1, rho=rho, s=s, l=l, b=b, beta_2=beta_2)

{'beta_1_deg': np.float64(53.99243843274894),
 'beta_2_deg': np.float64(50.21),
 'beta_m_deg': np.float64(52.181469884406404),
 'W_m': np.float64(16.308888606198593),
 'L_force': np.float64(0.23205630394936233),
 'D_force': np.float64(0.013559933530853552),
 'C_L': np.float64(0.40391584186693424),
 'C_D': np.float64(0.023602340787818244),
 'DF': np.float64(0.18446273039604505)}